# Traffic Sign Label-Flipping Poisoning Assessment

In this exercise, you will assess how simple label flipping can affect a traffic sign classifier. The attack changes a fraction of `Stop` sign training labels to `Yield`, retrains the model, and measures whether clean stop signs are more likely to be misclassified as yield signs.

Unlike a triggered backdoor, this workflow does not change image pixels. The poisoned samples look visually clean, but their labels are wrong.

In [ ]:
from pathlib import Path
import os
import sys

CURRENT = Path.cwd().resolve()
ROOT = CURRENT.parent if CURRENT.name == "notebooks" else CURRENT
sys.path.append(str(ROOT / "src"))

os.environ["ART_DATA_PATH"] = str(ROOT / "art_data")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".matplotlib")

import numpy as np
import torch

from traffic_sign_poisoning_utils import (
    SELECTED_CLASS_NAMES,
    SOURCE_CLASS_ID,
    SOURCE_CLASS_NAME,
    TARGET_CLASS_ID,
    TARGET_CLASS_NAME,
    build_traffic_sign_cnn,
    class_accuracy_table,
    create_label_flip_poisoned_training_set,
    evaluate_clean,
    plot_label_flip_examples,
    prepare_gtsrb_subsets,
    save_checkpoint,
    targeted_misclassification_rate,
    train_model,
    write_results_table,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_DIR = ROOT / "data" / "generated"
DOWNLOAD_DIR = ROOT / "data" / "gtsrb"
MODEL_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(7)
np.random.seed(7)

print(f"Device: {DEVICE}")
print(SELECTED_CLASS_NAMES)

## 1. Load the Traffic Sign Dataset

The utility builds a compact GTSRB subset using six classes from the stop-sign security scenario. The first run downloads GTSRB through torchvision.

In [ ]:
prepare_gtsrb_subsets(DATA_DIR, DOWNLOAD_DIR, train_per_class=80, val_per_class=30)
train_data = np.load(DATA_DIR / "traffic_sign_train_clean.npz")
val_data = np.load(DATA_DIR / "traffic_sign_val_clean.npz")

train_images, train_labels = train_data["images"], train_data["labels"]
val_images, val_labels = val_data["images"], val_data["labels"]

print(train_images.shape, train_labels.shape)
print(val_images.shape, val_labels.shape)
print(f"Source class: {SOURCE_CLASS_NAME} -> target class: {TARGET_CLASS_NAME}")

## 2. Train and Evaluate a Clean Baseline

This section calls the targeted metric helper. If you have not completed `targeted_misclassification_rate` yet, complete the TODO in `src/traffic_sign_poisoning_utils.py` before running the final metric lines in this cell.

In [ ]:
clean_model = build_traffic_sign_cnn()
clean_model = train_model(clean_model, train_images, train_labels, device=DEVICE, epochs=4)
save_checkpoint(clean_model, MODEL_DIR / "clean_traffic_sign_cnn.pt")

clean_eval = evaluate_clean(clean_model, val_images, val_labels, device=DEVICE)
clean_source_accuracy = [row for row in class_accuracy_table(clean_eval["predictions"], val_labels) if row["class_id"] == SOURCE_CLASS_ID][0]["accuracy"]
clean_targeted_rate = targeted_misclassification_rate(clean_eval["predictions"], val_labels, SOURCE_CLASS_ID, TARGET_CLASS_ID)

print(clean_eval["accuracy"], clean_source_accuracy, clean_targeted_rate)

## 3. Implement Label-Flipping Poisoning

Open `src/traffic_sign_poisoning_utils.py` and complete:

- `create_label_flip_poisoned_training_set`
- `targeted_misclassification_rate`

Then rerun the cells below.

In [ ]:
POISON_FRACTION = 0.75

poisoned_images, poisoned_labels, flipped_indices = create_label_flip_poisoned_training_set(
    train_images,
    train_labels,
    source_class=SOURCE_CLASS_ID,
    target_class=TARGET_CLASS_ID,
    poison_fraction=POISON_FRACTION,
)

print(f"Flipped labels: {len(flipped_indices)}")
print(f"Original source-class training samples: {np.sum(train_labels == SOURCE_CLASS_ID)}")
print(f"Poisoned labels now marked as target: {np.sum(poisoned_labels[flipped_indices] == TARGET_CLASS_ID)}")

In [ ]:
plot_label_flip_examples(
    train_images,
    train_labels,
    poisoned_labels,
    flipped_indices,
    RESULTS_DIR / "label_flip_examples.png",
)

## 4. Retrain on the Poisoned Dataset

In [ ]:
poisoned_model = build_traffic_sign_cnn()
poisoned_model = train_model(poisoned_model, poisoned_images, poisoned_labels, device=DEVICE, epochs=4)
save_checkpoint(poisoned_model, MODEL_DIR / "label_flipped_traffic_sign_cnn.pt")

poisoned_eval = evaluate_clean(poisoned_model, val_images, val_labels, device=DEVICE)
poisoned_source_accuracy = [row for row in class_accuracy_table(poisoned_eval["predictions"], val_labels) if row["class_id"] == SOURCE_CLASS_ID][0]["accuracy"]
poisoned_targeted_rate = targeted_misclassification_rate(poisoned_eval["predictions"], val_labels, SOURCE_CLASS_ID, TARGET_CLASS_ID)

print(poisoned_eval["accuracy"], poisoned_source_accuracy, poisoned_targeted_rate)

## 5. Compare Clean and Poisoned Behavior

Focus on the source class and targeted misclassification rate. A poisoned model may look acceptable if you only inspect aggregate validation accuracy.

In [ ]:
rows = [
    {
        "model": "clean_baseline",
        "clean_accuracy": round(clean_eval["accuracy"], 4),
        "source_class_accuracy": round(clean_source_accuracy, 4),
        "targeted_misclassification_rate": round(clean_targeted_rate, 4),
        "mean_confidence": round(clean_eval["mean_confidence"], 4),
        "notes": "trained_on_clean_labels",
    },
    {
        "model": "label_flipped_model",
        "clean_accuracy": round(poisoned_eval["accuracy"], 4),
        "source_class_accuracy": round(poisoned_source_accuracy, 4),
        "targeted_misclassification_rate": round(poisoned_targeted_rate, 4),
        "mean_confidence": round(poisoned_eval["mean_confidence"], 4),
        "notes": f"{POISON_FRACTION:.0%}_of_source_labels_flipped_to_target",
    },
]

write_results_table(rows, RESULTS_DIR / "label_flip_metrics.csv")
rows

## 6. Assessment Report

Complete `docs/assessment_report_template.md` using your results.

Include:

- Whether aggregate accuracy changed enough to reveal the issue.
- Whether source-class accuracy changed.
- Whether the targeted misclassification rate increased.
- Why label provenance and class-specific validation matter for traffic sign systems.